# Práctica: Contrastes


In [2]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# Cargar datos:
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv("/content/drive/MyDrive/archive/datos_limpios.csv")

Mounted at /content/drive


### Variables de nuestra base de datos
- `exam_score`: Calificación obtenida en el examen (variable dependiente).  
- `hours_studied`: Horas estudiadas para el examen.  
- `sleep_hours`: Horas de sueño al día.  
- `attendance_percent`: Porcentaje de asistencia a clase.  
- `previous_scores`: Calificaciones obtenidas anteriormente.


In [3]:
# Comprobar nombres exactos de columnas en su instalación
df.columns


Index(['hours_studied', 'sleep_hours', 'attendance_percent', 'previous_scores',
       'exam_score', 'student_id_S002', 'student_id_S003', 'student_id_S004',
       'student_id_S005', 'student_id_S006',
       ...
       'student_id_S191', 'student_id_S192', 'student_id_S193',
       'student_id_S194', 'student_id_S195', 'student_id_S196',
       'student_id_S197', 'student_id_S198', 'student_id_S199',
       'student_id_S200'],
      dtype='object', length=204)

## 1) Ajuste del modelo OLS


In [4]:
y = df["exam_score"]

X_cols = ['hours_studied', 'sleep_hours', 'attendance_percent', 'previous_scores']
X = df[X_cols].astype(float)


# Ajustar OLS
model = sm.OLS(y, sm.add_constant(X))
results = model.fit()
print(results.summary())


                            OLS Regression Results                            
Dep. Variable:             exam_score   R-squared:                       0.841
Model:                            OLS   Adj. R-squared:                  0.838
Method:                 Least Squares   F-statistic:                     258.7
Date:                Fri, 28 Nov 2025   Prob (F-statistic):           8.76e-77
Time:                        15:49:19   Log-Likelihood:                -482.21
No. Observations:                 200   AIC:                             974.4
Df Residuals:                     195   BIC:                             990.9
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -2.1421      1

## 2) Significación **global**

La prueba F en el resumen contrasta:  
$$
H_0: \beta_1=\beta_2=\cdots=\beta_k=0
$$
frente a la alternativa de que **algún** coeficiente es no nulo.


In [5]:
print("Estadístico F global:", results.fvalue)
print("p-valor F global:", results.f_pvalue)

Estadístico F global: 258.6735638313304
p-valor F global: 8.758592533735377e-77


El **test de significación global F** consiste en comprobar si todas las estimaciones son igual a 0, es decir, explica que ninguna de las variables explicativas `(horas estudiadas, horas de sueño, porcentaje de asistencia, notas previas)` tienen relación lineal con la nota del examen.

Si el estadístico F global fuera suficientemente mayor que 0 y su p valor muy pequeño, entonces se rechazaría H0 y se puede afirmar que alguno de los coeficientes es distinto de 0, por lo que el modelo tiene capacidad explicativa.

Como se ha obtenido un estadístico F global igual a 258.6735638313304 y un p-valor 8.758592533735377e-77, **se rechaza la hipótesis nula** y podemos afirmar que **al menos una de las variables contribuye de manera significativa** a modelar la nota del examen.

En resumen, **el modelo presenta un ajuste globalmente significativo**.

## 3) Significación **individual**

Para cada \( \beta_j \):  
$$
H_0: \beta_j = 0
$$


In [6]:
coef_table = pd.DataFrame({
    "coef": results.params,
    "std_err": results.bse,
    "t": results.tvalues,
    "p>|t|": results.pvalues
})
coef_table


,coef,std_err,t,p>|t|
const,-2.142087,1.670793,-1.282078,2.013375e-01
hours_studied,1.555260,0.060440,25.732452,1.308141e-64
sleep_hours,0.952258,0.132426,7.190851,1.343306e-11
attendance_percent,0.108390,0.013615,7.960877,1.376740e-13
previous_scores,0.177285,0.012668,13.995148,2.884605e-31


El objetivo del test es contrastar si un coeficiente de regresión ($\beta_k$) es significativamente distinto de cero. Si un coeficiente es igual a cero, significa que la variable independiente asociada a ese coeficiente no tiene un efecto lineal sobre la variable dependiente.

**Formulación de las Hipótesis:**
Para cada coeficiente $\beta_k$ (donde $k=1, 2, 3, \dots, K$ para las variables explicativas), se plantea la siguiente hipótesis:

*Hipótesis Nula ($H_0$):* El coeficiente es igual a cero.$$H_0: \beta_k = 0$$
*Hipótesis Alternativa ($H_1$):* El coeficiente es distinto de cero (es significativo).$$H_1: \beta_k \neq 0$$
**Cálculo del Estadístico** El estadístico $t$ se calcula para cada coeficiente usando la siguiente fórmula. Esta medida indica cuántas desviaciones estándar (errores estándar) se encuentra el coeficiente estimado de cero:$$t  = \frac{\hat{\beta}_k}{\sqrt{\hat {var}}[\hat{\beta}_k]}$$

Usaremos para decidir el p valor:

Rechazar $H_0$$\text{Valor P} < \alpha$. La variable es estadísticamente significativa.

No Rechazar $H_0$$\text{Valor P} \geq \alpha$. La variable es estadísticamente NO significativa.

Podemos comprobar que para todas y cada una de las variables de nuestro modelo, el P valor es muy pequeño (próximo a 0) por lo que, rechazamos $H_0$y en por tanto afirmar que todas las variables son individualmente significativas con una confianza de α = 5.

## 4) Contrastes sobre **relaciones lineales** \(R\beta = r\)

Además de probar si un coeficiente es 0, a menudo queremos saber si **dos efectos son iguales** o si **una combinación** de efectos cumple cierta relación.


In [7]:
# Ver el orden de parámetros (índices de beta)
results.params.index


Index(['const', 'hours_studied', 'sleep_hours', 'attendance_percent',
       'previous_scores'],
      dtype='object')

### 4.1) Igualdad de dos coeficientes: $\beta_{\text{hours_studied}} = \beta_{\text{sleep_hours}}$

Interpretación: “¿Un incremento marginal en las horas estudiadas (hours_studied) tiene el **mismo** efecto sobre el empleo que un incremento marginal en las horas diarias de sueño (sleep_hours)?”


In [8]:

names = results.params.index.tolist()

# Construir restricción: beta_GNP - beta_UNEMP = 0
R = np.zeros((1, len(names)))
R[0, names.index('hours_studied')]   = 1.0
R[0, names.index('sleep_hours')] = -1.0
r = np.array([0.0])

ftest_eq = results.f_test((R, r))   # F-test de igualdad

print("== Igualdad de coeficientes: beta_hours_studied = beta_sleep_hours ==")
print("F-stat:", ftest_eq.fvalue, "p-value:", ftest_eq.pvalue)
ftest_eq


== Igualdad de coeficientes: beta_hours_studied = beta_sleep_hours ==
F-stat: 16.025212005156963 p-value: 8.877691154155896e-05


<class 'statsmodels.stats.contrast.ContrastResults'>
<F test: F=16.025212005156963, p=8.877691154155896e-05, df_denom=195, df_num=1>

Definición de Hipótesis:

Hipótesis Nula ($H_0$): $\beta_{\text{hours_studied}} = \beta_{\text{sleep_hours}}. $Interpretación: El efecto marginal de un incremento en las horas estudiadas sobre la variable dependiente (el empleo, según tu enunciado) es el mismo que el efecto marginal de un incremento en las horas de sueño.


Hipótesis Alternativa ($H_1$): $\beta_{\text{hours_studied}} \neq \beta_{\text{sleep_hours}}. $Interpretación: El efecto marginal de las horas estudiadas es diferente al efecto marginal de las horas de sueño.

Teniendo en cuenta que nuestro p valor es muy pequeño (próximo a 0), rechazamos $H_0$, afurmamos por tanto que el efecto marginal de las horas estudiadas es diferente al efecto marginal de las horas de sueño.

### 4.2) Grupo de coeficientes nulos: $\beta_{\text{GNP}} = \beta_{\text{UNEMP}} = 0$


In [9]:
R = np.zeros((2, len(names)))
r = np.zeros(2)

R[0, names.index('hours_studied')]   = 1.0           # beta_GNP = 0
R[1, names.index('sleep_hours')] = 1.0           # beta_UNEMP = 0

ftest_group_zero = results.f_test((R, r))
print("== Grupo cero: beta_hours_studied = beta_sleep_hours = 0 ==")
print("F-stat:", float(ftest_group_zero.fvalue), "p-value:", float(ftest_group_zero.pvalue))


== Grupo cero: beta_hours_studied = beta_sleep_hours = 0 ==
F-stat: 377.59271901462944 p-value: 8.751016325328172e-68


### 4.3) Combinación lineal: $\beta_{\text{GNPDEFL}} + \beta_{\text{POP}} = \beta_{\text{ARMED}}$



In [10]:

# Restricción: beta_GNPDEFL + beta_POP - beta_ARMED = 0
R = np.zeros((1, len(names)))
R[0, names.index("hours_studied")] = 1.0
R[0, names.index("attendance_percent")]  = 1.0
R[0, names.index("previous_scores")]  = -1.0
r = np.array([0.0])

ftest_combo = results.f_test((R, r))
print("== Combinación: beta_exam_score + beta_attendance_percent = beta_previous_scores ==")
print("F-stat:", float(ftest_combo.fvalue), "p-value:", float(ftest_combo.pvalue))


== Combinación: beta_exam_score + beta_attendance_percent = beta_previous_scores ==
F-stat: 524.4756347174671 p-value: 3.495435008607092e-57


# Cuantiles Tablas

In [11]:
from scipy.stats import t, f, chi2

In [12]:
t.ppf(1-0.025, 16 - 7)

np.float64(2.2621571628540993)

In [13]:
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:             exam_score   R-squared:                       0.841
Model:                            OLS   Adj. R-squared:                  0.838
Method:                 Least Squares   F-statistic:                     258.7
Date:                Fri, 28 Nov 2025   Prob (F-statistic):           8.76e-77
Time:                        15:49:19   Log-Likelihood:                -482.21
No. Observations:                 200   AIC:                             974.4
Df Residuals:                     195   BIC:                             990.9
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -2.1421      1

In [14]:
f.ppf(1-0.05, 6, 16-7)

np.float64(3.373753647039213)

In [15]:
sigma_gorro=results.scale

In [16]:
(16-7)*sigma_gorro/chi2.ppf(1-0.025, 16-7)

np.float64(3.5294455569254697)

In [17]:
(16-7)*sigma_gorro/chi2.ppf(0.025, 16-7)

np.float64(24.863014497660043)

In [18]:
sigma_gorro

np.float64(7.459980365260067)

In [19]:
np.sqrt(sigma_gorro)

np.float64(2.7312964623526437)